# Shadow-Ronin 4B — Production Kaggle Training Pipeline

**One training run → one merged Shadow-Ronin model → all GGUF quantizations → Hugging Face upload.**

Base weights are downloaded from **Qwen/Qwen3-4B** only for training. The exported model is named **Shadow-Ronin-4B** and is written to separate output directories; the base model is never copied into the final artifact directory.

This notebook:
- discovers `shadow_ronin_combined_production_dataset.jsonl` (and can fall back to any JSONL under `/kaggle/input`)
- validates and deduplicates records
- trains QLoRA/SFT once
- saves the LoRA adapter
- merges the adapter into the base model
- exports a Shadow-Ronin F16 GGUF
- builds a Shadow-Ronin-specific calibration corpus and importance matrix
- creates Q2_K, Q3_K_M, Q4_0, Q4_K_M, Q5_K_S, Q5_K_M, Q6_K and Q8_0
- runs smoke tests on the merged model and GGUFs
- optionally pushes the merged HF model and all GGUFs to Hugging Face

**Important:** the internal architecture must remain `qwen3` for Transformers/llama.cpp compatibility. User-facing repository names, model folders, output filenames, and model metadata use `Shadow-Ronin-4B`.


In [ ]:
# =========================
# 1. CONFIGURATION
# =========================
import os, json, re, sys, subprocess, shutil, hashlib, textwrap, time
from pathlib import Path

BASE_MODEL = "Qwen/Qwen3-4B"

MODEL_NAME = "Shadow-Ronin-4B"
HF_USERNAME = os.environ.get("HF_USERNAME", "YOUR_HF_USERNAME")

# Set to True after adding HF_TOKEN to Kaggle Secrets/environment.
PUSH_TO_HF = True

HF_MODEL_REPO = f"{HF_USERNAME}/{MODEL_NAME}"
HF_GGUF_REPO = f"{HF_USERNAME}/{MODEL_NAME}-GGUF"

DATASET_FILENAME = "shadow_ronin_combined_production_dataset.jsonl"

WORK = Path("/kaggle/working/shadow_ronin")
ADAPTER_DIR = WORK / "adapter"
MERGED_DIR = WORK / "merged_hf"
GGUF_DIR = WORK / "gguf"
CALIB_DIR = WORK / "calibration"
LLAMA_CPP_DIR = WORK / "llama.cpp"
EVAL_DIR = WORK / "evaluation"

for d in [WORK, ADAPTER_DIR, MERGED_DIR, GGUF_DIR, CALIB_DIR, EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# QLoRA training settings
MAX_SEQ_LENGTH = 2048
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LEARNING_RATE = 1e-4
NUM_EPOCHS = 3
PER_DEVICE_BATCH = 1
GRAD_ACCUM = 8
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
SEED = 3407

# GGUF targets
QUANTS = [
    ("Q2_K", "Q2_K"),
    ("Q3_K_M", "Q3_K_M"),
    ("Q4_0", "Q4_0"),
    ("Q4_K_M", "Q4_K_M"),
    ("Q5_K_S", "Q5_K_S"),
    ("Q5_K_M", "Q5_K_M"),
    ("Q6_K", "Q6_K"),
    ("Q8_0", "Q8_0"),
]

print("Base:", BASE_MODEL)
print("Model:", MODEL_NAME)
print("HF model repo:", HF_MODEL_REPO)
print("HF GGUF repo:", HF_GGUF_REPO)
print("Working directory:", WORK)


In [ ]:
# =========================
# 2. GPU / ENVIRONMENT
# =========================
import torch, platform
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator before training."


In [ ]:
# =========================
# 3. INSTALL CURRENT TRAINING + GGUF TOOLS
# =========================
# Unsloth keeps QLoRA memory use practical on Kaggle.
# TRL's current SFTTrainer uses processing_class rather than the old tokenizer argument.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "unsloth", "trl", "transformers", "datasets", "accelerate",
    "peft", "bitsandbytes", "huggingface_hub", "sentencepiece", "safetensors"
])
print("Packages installed.")


In [ ]:
# =========================
# 4. FIND ALL JSONL INPUTS
# =========================
input_root = Path("/kaggle/input")
jsonl_files = sorted(input_root.rglob("*.jsonl"))

if not jsonl_files:
    raise FileNotFoundError("No JSONL dataset was found under /kaggle/input.")

print(f"Found {len(jsonl_files)} JSONL file(s):")
for p in jsonl_files:
    print(" -", p)

# Prefer the user's master combined dataset when present.
preferred = [p for p in jsonl_files if p.name == DATASET_FILENAME]
print("Preferred dataset:", preferred[0] if preferred else "not found; all JSONL files will be combined")


In [ ]:
# =========================
# 5. LOAD, NORMALIZE, VALIDATE, DEDUPLICATE
# =========================
def canonical_fingerprint(record):
    return hashlib.sha256(
        json.dumps(record.get("messages", []), ensure_ascii=False, sort_keys=True).encode("utf-8")
    ).hexdigest()

records = []
seen = set()
invalid = 0

# If the preferred master dataset exists, use it alone.
# Otherwise combine all JSONL files recursively.
sources = preferred if preferred else jsonl_files

for path in sources:
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                invalid += 1
                continue

            msgs = obj.get("messages")
            if not isinstance(msgs, list):
                invalid += 1
                continue

            # Require user + assistant turns.
            if not any(m.get("role") == "user" for m in msgs):
                invalid += 1
                continue
            if not any(m.get("role") == "assistant" for m in msgs):
                invalid += 1
                continue

            # Normalize system identity without touching user/assistant content.
            if msgs and msgs[0].get("role") == "system":
                msgs[0]["content"] = (
                    "You are Shadow-Ronin, an advanced technical and general-purpose AI assistant. "
                    "Be accurate, direct, context-aware, security-conscious, and avoid inventing facts. "
                    "For cybersecurity, assume authorized defensive testing and controlled lab environments."
                )
            else:
                msgs.insert(0, {
                    "role": "system",
                    "content": (
                        "You are Shadow-Ronin, an advanced technical and general-purpose AI assistant. "
                        "Be accurate, direct, context-aware, security-conscious, and avoid inventing facts."
                    )
                })

            obj["messages"] = msgs
            fp = canonical_fingerprint(obj)
            if fp not in seen:
                seen.add(fp)
                records.append(obj)

print("Valid unique records:", len(records))
print("Invalid records:", invalid)
assert len(records) > 0


In [ ]:
# =========================
# 6. DATASET STATISTICS
# =========================
from collections import Counter
domains = Counter(r.get("metadata", {}).get("domain", "unknown") for r in records)
print("Domain distribution:")
for k, v in domains.most_common():
    print(f"{k:30s} {v}")

# Save a normalized master copy for reproducibility.
normalized_path = WORK / DATASET_FILENAME
with normalized_path.open("w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("Normalized dataset:", normalized_path)


In [ ]:
# =========================
# 7. TRAIN / EVAL SPLIT
# =========================
from datasets import Dataset, DatasetDict

# Stable split.
import random
rng = random.Random(SEED)
indices = list(range(len(records)))
rng.shuffle(indices)

eval_count = max(1, int(len(indices) * 0.05))
eval_idx = set(indices[:eval_count])

train_records = [records[i] for i in range(len(records)) if i not in eval_idx]
eval_records  = [records[i] for i in range(len(records)) if i in eval_idx]

train_ds = Dataset.from_list(train_records)
eval_ds = Dataset.from_list(eval_records)

print("Train:", len(train_ds))
print("Eval :", len(eval_ds))


In [ ]:
# =========================
# 8. LOAD QWEN3-4B FROM HUGGING FACE
# =========================
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

print("Loaded:", BASE_MODEL)


In [ ]:
# =========================
# 9. FORMAT CHAT DATA
# =========================
def format_messages(example):
    # Qwen3 chat template is kept from the base tokenizer.
    # enable_thinking=False makes this SFT dataset train direct answers rather than hidden reasoning traces.
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    return {"text": text}

train_text = train_ds.map(format_messages, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_messages, remove_columns=eval_ds.column_names)

print(train_text[0]["text"][:1500])


In [ ]:
# =========================
# 10. APPLY LoRA / QLoRA
# =========================
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

print("LoRA configured.")


In [ ]:
# =========================
# 11. SFT TRAINING
# =========================
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=str(ADAPTER_DIR),
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    load_best_model_at_end=False,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=1.0,
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    seed=SEED,
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    args=training_args,
)

print("Starting training...")
stats = trainer.train()
print(stats)


In [ ]:
# =========================
# 12. SAVE LoRA ADAPTER
# =========================
ADAPTER_FINAL = ADAPTER_DIR / "Shadow-Ronin-4B-LoRA"
ADAPTER_FINAL.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(ADAPTER_FINAL))
tokenizer.save_pretrained(str(ADAPTER_FINAL))

print("LoRA adapter saved:", ADAPTER_FINAL)


In [ ]:
# =========================
# 13. MERGE LoRA INTO FULL SHADOW-RONIN MODEL
# =========================
# Reload the original HF base only as an internal merge source.
# It is NOT copied into the final output directory.
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

del trainer, model
import gc
gc.collect()
torch.cuda.empty_cache()

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

peft_model = PeftModel.from_pretrained(base_model, str(ADAPTER_FINAL))
merged_model = peft_model.merge_and_unload()

# User-facing identity. Do NOT change config["model_type"]; Qwen3 is required for compatibility.
merged_model.config._name_or_path = MODEL_NAME
base_tokenizer.name_or_path = MODEL_NAME

MERGED_FINAL = MERGED_DIR / MODEL_NAME
MERGED_FINAL.mkdir(parents=True, exist_ok=True)

merged_model.save_pretrained(
    str(MERGED_FINAL),
    safe_serialization=True,
    max_shard_size="4GB",
)
base_tokenizer.save_pretrained(str(MERGED_FINAL))

# Write an explicit model identity file.
(MERGED_FINAL / "SHADOW_RONIN_MODEL.txt").write_text(
    f"{MODEL_NAME}\nBase architecture: Qwen3-4B\nFine-tuning: QLoRA/SFT\n"
    "This directory contains the merged Shadow-Ronin model, not the original base checkpoint.\n",
    encoding="utf-8"
)

print("Merged model:", MERGED_FINAL)


In [ ]:
# =========================
# 14. QUICK TRANSFORMERS SMOKE TEST
# =========================
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

test_tokenizer = AutoTokenizer.from_pretrained(str(MERGED_FINAL))
test_model = AutoModelForCausalLM.from_pretrained(
    str(MERGED_FINAL),
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
)

tests = [
    "Who are you and what is your model name?",
    "BigFix relevance lo oka file exists aa ani ela check chestaru?",
    "Explain how to investigate a suspicious RDP login.",
    "Python function ni code review chesi bugs identify cheyyi.",
    "2x² - 7x + 3 = 0 solve cheyyi.",
    "Linux server lo suspicious process kanipiste first em check cheyyali?",
]

for prompt in tests:
    messages = [{"role": "user", "content": prompt}]
    text = test_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = test_tokenizer(text, return_tensors="pt").to(test_model.device)
    with torch.no_grad():
        out = test_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            temperature=None,
            top_p=None,
        )
    answer = test_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print("\nUSER:", prompt)
    print("SHADOW-RONIN:", answer[:1500])

del test_model
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# =========================
# 15. BUILD LLAMA.CPP
# =========================
if LLAMA_CPP_DIR.exists() and not (LLAMA_CPP_DIR / ".git").exists():
    shutil.rmtree(LLAMA_CPP_DIR)

if not LLAMA_CPP_DIR.exists():
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/ggml-org/llama.cpp.git",
        str(LLAMA_CPP_DIR)
    ])

build_dir = LLAMA_CPP_DIR / "build"
subprocess.check_call([
    "cmake", "-S", str(LLAMA_CPP_DIR), "-B", str(build_dir),
    "-DGGML_CUDA=ON", "-DCMAKE_BUILD_TYPE=Release"
])
subprocess.check_call([
    "cmake", "--build", str(build_dir), "--config", "Release",
    "-j", str(max(2, os.cpu_count() or 2))
])

print("llama.cpp built.")


In [ ]:
# =========================
# 16. CONVERT MERGED HF MODEL -> F16 GGUF
# =========================
convert_script = LLAMA_CPP_DIR / "convert_hf_to_gguf.py"
F16_GGUF = GGUF_DIR / f"{MODEL_NAME}-F16.gguf"

# llama.cpp's converter reads the merged HF directory directly.
subprocess.check_call([
    sys.executable, str(convert_script),
    str(MERGED_FINAL),
    "--outfile", str(F16_GGUF),
    "--outtype", "f16"
])

assert F16_GGUF.exists()
print("F16 GGUF:", F16_GGUF)


In [ ]:
# =========================
# 17. BUILD SHADOW-RONIN CALIBRATION CORPUS
# =========================
# Use actual Shadow-Ronin domain data rather than a generic calibration set.
calib_path = CALIB_DIR / "shadow_ronin_calibration.txt"

# Stratify by domain and keep representative text.
by_domain = {}
for r in records:
    d = r.get("metadata", {}).get("domain", "unknown")
    by_domain.setdefault(d, []).append(r)

lines = []
for domain in sorted(by_domain):
    for r in by_domain[domain][:50]:
        for m in r["messages"]:
            if m["role"] in ("user", "assistant"):
                content = m["content"].strip()
                if content:
                    lines.append(f"[{domain}] {content}")

# Bound calibration size to keep imatrix practical.
lines = lines[:5000]
calib_path.write_text("\n\n".join(lines), encoding="utf-8")

print("Calibration examples:", len(lines))
print("Calibration file:", calib_path)


In [ ]:
# =========================
# 18. GENERATE IMPORTANCE MATRIX
# =========================
imatrix_bin = build_dir / "bin" / "llama-imatrix"
if not imatrix_bin.exists():
    imatrix_bin = build_dir / "bin" / "llama-imatrix"

IMATRIX = CALIB_DIR / "shadow_ronin.imatrix.gguf"

subprocess.check_call([
    str(imatrix_bin),
    "-m", str(F16_GGUF),
    "-f", str(calib_path),
    "-o", str(IMATRIX),
    "-ngl", "99",
])

assert IMATRIX.exists()
print("Importance matrix:", IMATRIX)


In [ ]:
# =========================
# 19. QUANTIZE ALL FINAL GGUF VARIANTS
# =========================
quant_bin = build_dir / "bin" / "llama-quantize"
if not quant_bin.exists():
    raise FileNotFoundError(quant_bin)

created = []

for label, qtype in QUANTS:
    out_file = GGUF_DIR / f"{MODEL_NAME}-{label}.gguf"

    cmd = [
        str(quant_bin),
        "--imatrix", str(IMATRIX),
        str(F16_GGUF),
        str(out_file),
        qtype,
        str(max(2, os.cpu_count() or 2)),
    ]

    print("\nQuantizing:", label)
    subprocess.check_call(cmd)

    if not out_file.exists() or out_file.stat().st_size == 0:
        raise RuntimeError(f"Quantization failed: {label}")

    created.append(out_file)
    print("Created:", out_file, "size GiB:", round(out_file.stat().st_size / 1024**3, 3))

print("\nAll quantizations complete.")


In [ ]:
# =========================
# 20. FINAL ARTIFACT CHECK
# =========================
for p in sorted(GGUF_DIR.glob("*.gguf")):
    print(f"{p.name:45s} {p.stat().st_size / 1024**3:8.3f} GiB")

expected = [
    f"{MODEL_NAME}-F16.gguf",
    *[f"{MODEL_NAME}-{q}.gguf" for q, _ in QUANTS]
]

missing = [x for x in expected if not (GGUF_DIR / x).exists()]
assert not missing, f"Missing GGUFs: {missing}"
print("Artifact check: PASS")


In [ ]:
# =========================
# 21. GGUF SMOKE TESTS
# =========================
llama_cli = build_dir / "bin" / "llama-cli"
if not llama_cli.exists():
    # Some builds place it directly in build/bin; this should catch the normal case.
    raise FileNotFoundError(llama_cli)

prompt = "You are Shadow-Ronin. In one concise paragraph, explain why network segmentation reduces security blast radius."

for gguf in sorted(GGUF_DIR.glob("*.gguf")):
    if gguf.name.endswith("-F16.gguf") or gguf.name.endswith(("Q2_K.gguf","Q3_K_M.gguf","Q4_K_M.gguf")):
        print("\nTesting:", gguf.name)
        result = subprocess.run(
            [
                str(llama_cli), "-m", str(gguf),
                "-p", prompt,
                "-n", "120",
                "--temp", "0.2",
                "--no-display-prompt",
            ],
            text=True, capture_output=True, timeout=180
        )
        output = (result.stdout or "")[-2000:]
        print(output)
        if result.returncode != 0:
            print(result.stderr[-1000:])
            raise RuntimeError(f"GGUF smoke test failed: {gguf.name}")

print("GGUF smoke tests completed.")


In [ ]:
# =========================
# 22. CREATE MODEL CARD / MANIFEST
# =========================
model_card = f"""---
library_name: transformers
base_model: {BASE_MODEL}
tags:
- shadow-ronin
- qwen3
- text-generation
- code
- cybersecurity
- bigfix
- linux
- devops
- telugu
- multilingual
---

# Shadow-Ronin 4B

Shadow-Ronin is a QLoRA/SFT fine-tuned model based on the Qwen3-4B architecture.

## Training
- Base: {BASE_MODEL}
- Dataset: {DATASET_FILENAME}
- Max sequence length: {MAX_SEQ_LENGTH}
- LoRA rank: {LORA_R}
- LoRA alpha: {LORA_ALPHA}
- Epochs: {NUM_EPOCHS}
- Learning rate: {LEARNING_RATE}
- Seed: {SEED}

## Domains
Telugu/Roman Telugu, English, mathematics, programming, software architecture, cloud, DevOps, networking, cybersecurity, pentesting methodology, malware analysis, ransomware defense, RDP security, BigFix, CIS, DISA STIG, SCAP, Linux, code review, debugging, repository intelligence, philosophy, geography and science.

## Artifacts
The GGUF repository contains:
Q2_K, Q3_K_M, Q4_0, Q4_K_M, Q5_K_S, Q5_K_M, Q6_K, Q8_0 and F16.

## Naming
All exported artifacts use the Shadow-Ronin name. The underlying `qwen3` architecture identifier is retained because it is required for compatible loading.

## Security scope
Cybersecurity examples are intended for authorized defensive testing and controlled laboratory environments.
"""
(MERGED_FINAL / "README.md").write_text(model_card, encoding="utf-8")
(GGUF_DIR / "README.md").write_text(model_card, encoding="utf-8")
print(model_card)


In [ ]:
# =========================
# 23. OPTIONAL: PUSH TO HUGGING FACE
# =========================
# Add HF_TOKEN as a Kaggle Secret or environment variable before running this cell.
from huggingface_hub import HfApi, create_repo

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")

if not PUSH_TO_HF:
    print("PUSH_TO_HF=False; skipping upload.")
elif not HF_TOKEN or HF_USERNAME == "YOUR_HF_USERNAME":
    print("Upload skipped. Set HF_USERNAME and HF_TOKEN, then rerun this cell.")
else:
    api = HfApi(token=HF_TOKEN)

    # Transformers model repo: merged Shadow-Ronin only.
    create_repo(HF_MODEL_REPO, token=HF_TOKEN, exist_ok=True, repo_type="model")
    api.upload_folder(
        folder_path=str(MERGED_FINAL),
        repo_id=HF_MODEL_REPO,
        repo_type="model",
        token=HF_TOKEN,
        commit_message="Upload Shadow-Ronin 4B merged model"
    )

    # GGUF repo: all quantizations only.
    create_repo(HF_GGUF_REPO, token=HF_TOKEN, exist_ok=True, repo_type="model")
    api.upload_folder(
        folder_path=str(GGUF_DIR),
        repo_id=HF_GGUF_REPO,
        repo_type="model",
        token=HF_TOKEN,
        commit_message="Upload Shadow-Ronin 4B GGUF family"
    )

    print("Uploaded:")
    print(HF_MODEL_REPO)
    print(HF_GGUF_REPO)


In [ ]:
# =========================
# 24. FINAL INVENTORY
# =========================
print("\n================ SHADOW-RONIN FINAL INVENTORY ================\n")
print("HF model repo :", HF_MODEL_REPO)
print("HF GGUF repo  :", HF_GGUF_REPO)
print("\nLocal GGUF artifacts:")
for p in sorted(GGUF_DIR.glob("*.gguf")):
    print(f"  {p.name:40s} {p.stat().st_size / 1024**3:.3f} GiB")

print("\nModel identity:", MODEL_NAME)
print("Base used only for training/merge:", BASE_MODEL)
print("No base checkpoint is placed in the GGUF output directory.")
